# Hand Detection CNN — Google Colab Training Notebook

This notebook trains a **CNN regression model from scratch** to detect hands and predict bounding boxes `(x, y, w, h)`.

### Pipeline overview
1. Mount Google Drive
2. Install / verify dependencies
3. Use **MediaPipe Hands** to auto-label every image with a normalised bounding box
4. Build a memory-efficient **`tf.data`** pipeline
5. Split into **train / validation / test** (70 / 15 / 15) using TensorFlow only
6. Apply **data augmentation** on the training set
7. Define and compile a custom **CNN** (no pre-trained weights)
8. Train with **model checkpoints** saved after every epoch to Google Drive
9. Evaluate on the test set and visualise predictions

---
## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/MyDrive')

---
## Step 2 — Install / Verify Dependencies

In [ ]:
# Install MediaPipe (TensorFlow is pre-installed on Colab)
!pip install -q mediapipe opencv-python-headless

import sys
import importlib

# Verify imports
import tensorflow as tf
import mediapipe as mp
import cv2
import numpy as np
import os
import pathlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches

print(f'TensorFlow  : {tf.__version__}')
print(f'MediaPipe   : {mp.__version__}')
print(f'OpenCV      : {cv2.__version__}')
print(f'NumPy       : {np.__version__}')
print('GPU available:', tf.config.list_physical_devices("GPU"))

---
## Step 3 — Configuration

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
DATA_ROOT       = '/content/MyDrive/ML-CEP/data_128'
CHECKPOINT_DIR  = '/content/MyDrive/ML-CEP/modelForHands'
LABELS_CACHE    = '/content/labels_cache.npz'   # cached on local disk (fast)

# ── Image / model settings ───────────────────────────────────────────────────
IMG_SIZE        = 128          # images are already 128×128
BATCH_SIZE      = 32
EPOCHS          = 50
LEARNING_RATE   = 1e-3

# ── Split ratios ─────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# ── Subfolder names (person names) — must match exact folder names on Drive ──
PERSON_FOLDERS = ['manahil', 'sitwat', 'Marium', 'talha']  # 'Marium' is capitalised in the dataset

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Config OK')
print(f'  DATA_ROOT      : {DATA_ROOT}')
print(f'  CHECKPOINT_DIR : {CHECKPOINT_DIR}')

---
## Step 4 — Generate Bounding-Box Labels with MediaPipe

Images are processed **one at a time** so that RAM usage stays minimal.
Results are cached to disk so this step only needs to run once.

In [ ]:
def get_hand_bbox_mediapipe(image_bgr):
    """
    Run MediaPipe Hands on a BGR image.
    Returns normalised (x, y, w, h) of the tightest bounding box
    around all detected hand landmarks, or None if no hand found.
    """
    mp_hands   = mp.solutions.hands
    image_rgb  = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    h, w       = image_rgb.shape[:2]

    with mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=1,
        min_detection_confidence=0.3
    ) as hands:
        results = hands.process(image_rgb)

    if not results.multi_hand_landmarks:
        return None

    # Collect all landmark x / y values (normalised 0–1)
    xs = [lm.x for lm in results.multi_hand_landmarks[0].landmark]
    ys = [lm.y for lm in results.multi_hand_landmarks[0].landmark]

    x_min, x_max = max(0.0, min(xs)), min(1.0, max(xs))
    y_min, y_max = max(0.0, min(ys)), min(1.0, max(ys))

    # Add a small padding (5 % of image size)
    pad = 0.05
    x_min = max(0.0, x_min - pad)
    y_min = max(0.0, y_min - pad)
    x_max = min(1.0, x_max + pad)
    y_max = min(1.0, y_max + pad)

    bw = x_max - x_min
    bh = y_max - y_min

    return float(x_min), float(y_min), float(bw), float(bh)


def collect_image_paths(data_root, class_folders, person_folders):
    """Walk the dataset tree and return all image file paths."""
    paths = []
    data_root = pathlib.Path(data_root)
    for cls in class_folders:
        for person in person_folders:
            folder = data_root / cls / person
            if not folder.exists():
                continue
            for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp'):
                paths.extend(folder.glob(ext))
    return [str(p) for p in paths]


# ── Discover class (gesture) folders (1–26) ──────────────────────────────────
data_root_path  = pathlib.Path(DATA_ROOT)
class_folders   = sorted(
    [d.name for d in data_root_path.iterdir() if d.is_dir()],
    key=lambda x: int(x) if x.isdigit() else x
)
print(f'Found {len(class_folders)} class folders: {class_folders[:5]} …')

all_image_paths = collect_image_paths(DATA_ROOT, class_folders, PERSON_FOLDERS)
print(f'Total images discovered: {len(all_image_paths)}')

In [ ]:
# ── Run MediaPipe labelling (skipped if cache exists) ───────────────────────
if os.path.exists(LABELS_CACHE):
    print('Loading cached labels …')
    cache         = np.load(LABELS_CACHE, allow_pickle=True)
    valid_paths   = list(cache['paths'])
    valid_boxes   = list(cache['boxes'])
    print(f'  Loaded {len(valid_paths)} labelled images from cache.')
else:
    print('Running MediaPipe on all images (this may take a few minutes) …')
    valid_paths = []
    valid_boxes = []
    skipped     = 0

    for i, img_path in enumerate(all_image_paths):
        img = cv2.imread(img_path)
        if img is None:
            skipped += 1
            continue

        bbox = get_hand_bbox_mediapipe(img)
        if bbox is None:
            skipped += 1
            continue

        valid_paths.append(img_path)
        valid_boxes.append(bbox)

        if (i + 1) % 500 == 0:
            print(f'  Processed {i + 1} / {len(all_image_paths)} …')

    # Save cache
    np.savez(LABELS_CACHE,
             paths=np.array(valid_paths),
             boxes=np.array(valid_boxes, dtype=np.float32))
    print(f'\nDone!  Labelled: {len(valid_paths)}  |  Skipped: {skipped}')

valid_boxes_np = np.array(valid_boxes, dtype=np.float32)
print(f'Dataset size after filtering: {len(valid_paths)} images')
print(f'Box array shape             : {valid_boxes_np.shape}')

---
## Step 5 — Build a Memory-Efficient `tf.data` Pipeline

In [ ]:
# ── Convert path / box lists to tensors ──────────────────────────────────────
path_tensor = tf.constant(valid_paths,    dtype=tf.string)
box_tensor  = tf.constant(valid_boxes_np, dtype=tf.float32)

# ── Total size and split sizes ────────────────────────────────────────────────
N          = len(valid_paths)
n_train    = int(N * TRAIN_RATIO)
n_val      = int(N * VAL_RATIO)
n_test     = N - n_train - n_val

print(f'Total      : {N}')
print(f'Train      : {n_train}')
print(f'Validation : {n_val}')
print(f'Test       : {n_test}')

# ── Image loading / decoding function ────────────────────────────────────────
def load_image(path, box):
    """Read, decode, and normalise a single image."""
    raw   = tf.io.read_file(path)
    image = tf.image.decode_image(raw, channels=3, expand_animations=False)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    return image, box


# ── Data augmentation (applied to training set only) ─────────────────────────
def augment(image, box):
    """
    Apply random augmentations.
    Bounding-box coordinates are updated for horizontal flip.
    Other augmentations (brightness, contrast, saturation) don't
    affect the spatial bounding box.
    """
    # Random horizontal flip
    flip = tf.random.uniform(()) > 0.5
    image = tf.cond(flip, lambda: tf.image.flip_left_right(image), lambda: image)
    # Update box x coordinate: new_x = 1 - x - w
    x, y, w, h = box[0], box[1], box[2], box[3]
    new_x = tf.cond(flip, lambda: 1.0 - x - w, lambda: x)
    box   = tf.stack([new_x, y, w, h])

    # Random brightness  (±15 %)
    image = tf.image.random_brightness(image, max_delta=0.15)
    # Random contrast
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    # Random saturation
    image = tf.image.random_saturation(image, lower=0.8, upper=1.2)
    # Random hue shift
    image = tf.image.random_hue(image, max_delta=0.05)

    # Clip to [0, 1]
    image = tf.clip_by_value(image, 0.0, 1.0)

    # Random Gaussian noise (stddev=0.02)
    noise  = tf.random.normal(shape=tf.shape(image), mean=0.0, stddev=0.02)
    image  = tf.clip_by_value(image + noise, 0.0, 1.0)

    return image, box


# ── Shuffle the full index list deterministically ─────────────────────────────
SHUFFLE_BUFFER = 1000  # keep RAM usage low on Colab (full shuffle would require ~5000 entries)
indices        = tf.range(N)
shuffled_idx   = tf.random.shuffle(indices, seed=SEED)

train_idx = shuffled_idx[:n_train]
val_idx   = shuffled_idx[n_train : n_train + n_val]
test_idx  = shuffled_idx[n_train + n_val :]

train_paths = tf.gather(path_tensor, train_idx)
train_boxes = tf.gather(box_tensor,  train_idx)

val_paths   = tf.gather(path_tensor, val_idx)
val_boxes   = tf.gather(box_tensor,  val_idx)

test_paths  = tf.gather(path_tensor, test_idx)
test_boxes  = tf.gather(box_tensor,  test_idx)


# ── Build tf.data datasets ────────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(paths, boxes, augment_fn=None, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, boxes))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(paths), SHUFFLE_BUFFER), seed=SEED)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if augment_fn is not None:
        ds = ds.map(augment_fn, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)
    return ds


train_ds = make_dataset(train_paths, train_boxes, augment_fn=augment, shuffle=True)
val_ds   = make_dataset(val_paths,   val_boxes)
test_ds  = make_dataset(test_paths,  test_boxes)

print('Datasets ready.')
for images_batch, boxes_batch in train_ds.take(1):
    print(f'  Train batch — images: {images_batch.shape}  boxes: {boxes_batch.shape}')

---
## Step 6 — CNN Model Architecture

In [ ]:
from tensorflow.keras import layers, models, regularizers


def build_hand_detection_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    """
    Custom CNN for hand bounding-box regression.
    Output: 4 values → (x, y, w, h) all normalised to [0, 1].
    """
    inputs = layers.Input(shape=input_shape, name='input_image')

    # ── Block 1 ──────────────────────────────────────────────────────────────
    x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=regularizers.l2(1e-4))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2, 2))(x)   # 64×64
    x = layers.Dropout(0.25)(x)

    # ── Block 2 ──────────────────────────────────────────────────────────────
    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2, 2))(x)   # 32×32
    x = layers.Dropout(0.25)(x)

    # ── Block 3 ──────────────────────────────────────────────────────────────
    x = layers.Conv2D(128, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(128, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2, 2))(x)   # 16×16
    x = layers.Dropout(0.25)(x)

    # ── Block 4 ──────────────────────────────────────────────────────────────
    x = layers.Conv2D(256, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(256, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.3)(x)

    # ── Global pooling + head ─────────────────────────────────────────────────
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(512, use_bias=False,
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(128, use_bias=False,
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(0.3)(x)

    # Output: 4 values in [0, 1] — use Sigmoid activation
    outputs = layers.Dense(4, activation='sigmoid', name='bbox_output')(x)

    model = models.Model(inputs, outputs, name='HandDetectionCNN')
    return model


model = build_hand_detection_cnn()
model.summary()

---
## Step 7 — Custom IoU Metric and Loss

In [ ]:
# ── IoU metric (for monitoring) ───────────────────────────────────────────────
def iou_metric(y_true, y_pred):
    """
    Mean Intersection-over-Union for normalised (x, y, w, h) boxes.
    """
    # Convert (x, y, w, h) → (x1, y1, x2, y2)
    true_x1 = y_true[:, 0]
    true_y1 = y_true[:, 1]
    true_x2 = y_true[:, 0] + y_true[:, 2]
    true_y2 = y_true[:, 1] + y_true[:, 3]

    pred_x1 = y_pred[:, 0]
    pred_y1 = y_pred[:, 1]
    pred_x2 = y_pred[:, 0] + y_pred[:, 2]
    pred_y2 = y_pred[:, 1] + y_pred[:, 3]

    inter_x1 = tf.maximum(true_x1, pred_x1)
    inter_y1 = tf.maximum(true_y1, pred_y1)
    inter_x2 = tf.minimum(true_x2, pred_x2)
    inter_y2 = tf.minimum(true_y2, pred_y2)

    inter_w  = tf.maximum(0.0, inter_x2 - inter_x1)
    inter_h  = tf.maximum(0.0, inter_y2 - inter_y1)
    inter    = inter_w * inter_h

    area_true = y_true[:, 2] * y_true[:, 3]
    area_pred = y_pred[:, 2] * y_pred[:, 3]
    union     = area_true + area_pred - inter + 1e-7

    return tf.reduce_mean(inter / union)


# ── Combined loss: MSE + (1 - IoU) ───────────────────────────────────────────
def combined_bbox_loss(y_true, y_pred):
    mse_loss  = tf.reduce_mean(tf.square(y_true - y_pred))
    iou_loss  = 1.0 - iou_metric(y_true, y_pred)
    return mse_loss + 0.5 * iou_loss


print('Loss and metric functions defined.')

---
## Step 8 — Compile the Model

In [ ]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE,
    clipnorm=1.0          # gradient clipping for stability
)

model.compile(
    optimizer=optimizer,
    loss=combined_bbox_loss,
    metrics=[iou_metric, 'mae']
)

print('Model compiled.')

---
## Step 9 — Callbacks (Checkpoint, LR Scheduler, Early Stopping)

In [ ]:
# ── Checkpoint: save the full model after every epoch ────────────────────────
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(CHECKPOINT_DIR, 'epoch_{epoch:03d}_val_loss_{val_loss:.4f}.keras'),
    monitor='val_loss',
    save_best_only=False,    # save every epoch so we can resume
    save_weights_only=False,
    verbose=1
)

# ── Best-model checkpoint ─────────────────────────────────────────────────────
best_model_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(CHECKPOINT_DIR, 'best_model.keras'),
    monitor='val_loss',
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

# ── ReduceLROnPlateau ─────────────────────────────────────────────────────────
reduce_lr_cb = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

# ── Early stopping ────────────────────────────────────────────────────────────
early_stop_cb = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=12,
    restore_best_weights=True,
    verbose=1
)

# ── TensorBoard logging ───────────────────────────────────────────────────────
tensorboard_cb = tf.keras.callbacks.TensorBoard(
    log_dir=os.path.join(CHECKPOINT_DIR, 'tb_logs'),
    histogram_freq=1
)

callbacks_list = [
    checkpoint_cb,
    best_model_cb,
    reduce_lr_cb,
    early_stop_cb,
    tensorboard_cb,
]

print('Callbacks configured.')

---
## Step 10 — (Optional) Resume from Last Checkpoint

If your Colab session crashes, run this cell to reload the latest saved model.

In [ ]:
import glob as _glob

def find_latest_checkpoint(checkpoint_dir):
    """Return the path of the most recently saved epoch checkpoint, or None."""
    pattern  = os.path.join(checkpoint_dir, 'epoch_*.keras')
    files    = _glob.glob(pattern)
    if not files:
        return None
    # Sort by epoch number embedded in filename
    files.sort(key=lambda f: int(os.path.basename(f).split('_')[1]))
    return files[-1]


latest_ckpt = find_latest_checkpoint(CHECKPOINT_DIR)

if latest_ckpt is not None:
    print(f'Found checkpoint: {latest_ckpt}')
    model = tf.keras.models.load_model(
        latest_ckpt,
        custom_objects={
            'combined_bbox_loss': combined_bbox_loss,
            'iou_metric': iou_metric
        }
    )
    print('Model restored successfully. Continuing training …')
else:
    print('No checkpoint found. Starting training from scratch …')

---
## Step 11 — Train the Model

In [ ]:
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks_list,
    verbose=1
)

print('Training complete.')

---
## Step 12 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history.history['loss'],     label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Combined BBox Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

# IoU
axes[1].plot(history.history['iou_metric'],     label='Train IoU')
axes[1].plot(history.history['val_iou_metric'], label='Val IoU')
axes[1].set_title('IoU Metric')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

# MAE
axes[2].plot(history.history['mae'],     label='Train MAE')
axes[2].plot(history.history['val_mae'], label='Val MAE')
axes[2].set_title('Mean Absolute Error')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'training_curves.png'), dpi=150)
plt.show()

---
## Step 13 — Evaluate on the Test Set

In [ ]:
test_results = model.evaluate(test_ds, verbose=1)
metric_names = model.metrics_names
print('\n── Test Results ─────────────────────────────────')
for name, value in zip(metric_names, test_results):
    print(f'  {name:20s}: {value:.5f}')

---
## Step 14 — Visualise Predictions on Test Images

In [ ]:
def draw_box(ax, image_np, true_box, pred_box):
    """Draw ground-truth (green) and predicted (red) bounding boxes."""
    ax.imshow(image_np)
    h, w = image_np.shape[:2]

    for box, colour, label in [(true_box, 'lime', 'GT'),
                                (pred_box, 'red',  'Pred')]:
        x, y, bw, bh = box
        rect = patches.Rectangle(
            (x * w, y * h), bw * w, bh * h,
            linewidth=2, edgecolor=colour, facecolor='none',
            label=label
        )
        ax.add_patch(rect)
    ax.legend(fontsize=8, loc='upper right')
    ax.axis('off')


# Grab one batch from the test set
test_images, test_true_boxes = next(iter(test_ds))
test_pred_boxes = model.predict(test_images)

num_show = min(12, BATCH_SIZE)
cols     = 4
rows     = (num_show + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
axes_flat = axes.flatten()

for i in range(num_show):
    draw_box(
        axes_flat[i],
        test_images[i].numpy(),
        test_true_boxes[i].numpy(),
        test_pred_boxes[i]
    )

# Hide any unused axes
for j in range(num_show, len(axes_flat)):
    axes_flat[j].axis('off')

plt.suptitle('Ground Truth (green) vs Predicted (red) Bounding Boxes', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'predictions_visualisation.png'), dpi=150)
plt.show()

---
## Step 15 — Save Final Model

In [ ]:
final_model_path = os.path.join(CHECKPOINT_DIR, 'hand_detection_final.keras')
model.save(final_model_path)
print(f'Final model saved to: {final_model_path}')

# Also save in TensorFlow SavedModel format for deployment
saved_model_path = os.path.join(CHECKPOINT_DIR, 'hand_detection_savedmodel')
model.export(saved_model_path)
print(f'SavedModel exported to: {saved_model_path}')

---
## Step 16 — (Optional) Run Inference on a Single Image

In [ ]:
def predict_bounding_box(image_path_or_array, model):
    """
    Predict hand bounding box for a single 128×128 image.

    Parameters
    ----------
    image_path_or_array : str or np.ndarray
        File path OR a NumPy array (H, W, 3) with uint8 values.
    model : tf.keras.Model
        Trained hand detection model.

    Returns
    -------
    dict with keys 'x', 'y', 'w', 'h' (normalised 0-1)
    """
    if isinstance(image_path_or_array, str):
        img = cv2.imread(image_path_or_array)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        img = image_path_or_array

    img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE)).astype(np.float32) / 255.0
    img_batch   = np.expand_dims(img_resized, axis=0)  # shape (1, 128, 128, 3)

    pred = model.predict(img_batch, verbose=0)[0]       # shape (4,)
    x, y, w, h = float(pred[0]), float(pred[1]), float(pred[2]), float(pred[3])

    # ── Visualise ─────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.imshow(img_resized)
    rect = patches.Rectangle(
        (x * IMG_SIZE, y * IMG_SIZE), w * IMG_SIZE, h * IMG_SIZE,
        linewidth=2, edgecolor='red', facecolor='none'
    )
    ax.add_patch(rect)
    ax.set_title(f'Predicted: x={x:.3f}, y={y:.3f}, w={w:.3f}, h={h:.3f}')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

    return {'x': x, 'y': y, 'w': w, 'h': h}


# ── Example usage ─────────────────────────────────────────────────────────────
# Replace the path below with any 128×128 hand image on your Drive
example_image = valid_paths[0]   # first image in labelled set
result = predict_bounding_box(example_image, model)
print('Predicted bounding box:', result)